# Service evidence
Run from the project root in a Databricks Git folder with the sibling `config/services.yaml`. Use compute with PySpark and PyYAML.

In [0]:
%run ./00_config

In [0]:
from pathlib import Path
import yaml
from pyspark.sql import functions as F
from pyspark.sql.window import Window

config = yaml.safe_load(Path("config/services.yaml").read_text(encoding="utf-8"))
source = f"{OUT}.facilities_clean"
clean = spark.table(source)
clean.printSchema()
evidence_table = f"{OUT}.facility_service_evidence"
review_table = f"{OUT}.facility_service_review_v3"
print("Evidence source:", source, "| Services:", list(config["services"]))

In [0]:
import json
import re


def phrase_pattern(terms):
    # An empty list would compile to an empty alternation that matches between
    # punctuation marks, so it must never match instead.
    if not terms:
        return re.compile(r"(?!x)x")
    # Word boundaries avoid substring matches such as "MRI" in longer names.
    phrases = [r"\s+".join(re.escape(word) for word in t.split())
               for t in sorted(terms, key=len, reverse=True)]
    return re.compile(r"(?<!\w)(?:" + "|".join(phrases) + r")(?!\w)", re.I)


def compile_rules(config):
    rules = {}
    for service, terms in config["services"].items():
        rules[service] = {
            key: phrase_pattern(terms[key])
            for key in ("direct", "supporting", "specialties")
        }
        rules[service]["red_flags"] = phrase_pattern(config["red_flags"])
        rules[service]["service_veto"] = phrase_pattern(terms["red_flags"])
    return rules


def passages(value):
    # Preserve each quoted substring exactly, including capitalization.
    if isinstance(value, list):
        for item in value:
            yield from passages(item)
    elif isinstance(value, str):
        value = value.strip()
        if value.startswith("["):
            try:
                items = json.loads(value)
            except ValueError:
                return
            if isinstance(items, list):
                yield from passages(items)
            return
        for line in re.split(r"(?<=[.!?])\s+|[\r\n]+", value):
            if line.strip():
                yield line.strip()


def score_record(source_records, rules):
    evidence = []
    snippets = set()
    for record in source_records:
        for field, value in json.loads(record).items():
            snippets.update((field, line) for line in passages(value))

    for service, rule in rules.items():
        hits = []
        blocked = False
        for field, line in sorted(snippets):
            # Remove equipment phrases before testing direct terms. A machine name
            # alone does not document that the facility performs the procedure.
            remainder, equipment_hits = rule["supporting"].subn(" ", line)
            direct = bool(rule["direct"].search(remainder))
            specialty = bool(rule["specialties"].search(line))
            if rule["service_veto"].search(line):
                blocked = True
                continue
            if rule["red_flags"].search(line):
                # A service-specific contradiction vetoes even a positive duplicate.
                if direct or equipment_hits:
                    blocked = True
                continue
            if direct and field in {"procedure_list", "capability_list", "description"}:
                kind = "direct"
            elif equipment_hits or (field == "equipment_list" and direct):
                kind = "supporting"
            elif specialty:
                kind = "specialty"
            else:
                continue
            hits.append((kind, field, line))
        if blocked or not hits:
            continue
        direct_count = sum(kind == "direct" for kind, _, _ in hits)
        support_count = sum(kind == "supporting" for kind, _, _ in hits)
        if direct_count:
            level, score = "strong", 0.90 if support_count else 0.85
        elif support_count:
            level, score = "moderate", 0.55
        else:
            level, score = "weak", 0.20
        priority = {"direct": 0, "supporting": 1, "specialty": 2}
        hits.sort(key=lambda h: (priority[h[0]], h[1], h[2]))
        quotes = [(field, line) for _, field, line in hits]
        evidence.append((service, level, score, quotes, direct_count, support_count))
    return evidence

### Input and scoring
Rows sharing an ID contribute to one evidence record. Each distinct (field, sentence) counts once, with direct evidence taking priority within a sentence. Equipment alone is moderate, a specialty alone is weak, and an explicit procedure or capability is strong. Scores are rule weights, not probabilities. Red flags in a service mention suppress that pair, even if another row is positive. Missing evidence and suppressed pairs are omitted from ranking. Specialty-only matches may apply to multiple services.

In [0]:
id_col = "unique_id"
fields = ["procedure_list", "equipment_list", "capability_list", "description", "specialties_list"]
# Keep the original ID for downstream joins and skip placeholder IDs.
ids = F.col(id_col).cast("string")
valid_id = ids.isNotNull() & ~F.lower(F.trim(ids)).isin("", "null", "none", "n/a", "na")
documents = (clean.where(valid_id)
             .select(ids.alias("unique_id"), F.to_json(F.struct(*fields)).alias("record"))
             .groupBy("unique_id").agg(F.sort_array(F.collect_set("record")).alias("source_records")))

rules = compile_rules(config)
schema = """array<struct<service:string,evidence_level:string,evidence_score:double,
            quotes:array<struct<field:string,sentence:string>>,n_direct:int,n_supporting:int>>"""
score_udf = F.udf(lambda records: score_record(records, rules), schema)
evidence = (documents.select("unique_id", F.explode(score_udf("source_records")).alias("hit"))
            .select("unique_id", "hit.*"))
evidence.write.format("delta").mode("overwrite").saveAsTable(evidence_table)
published = spark.table(evidence_table)
display(published.groupBy("service", "evidence_level").count().orderBy("service", "evidence_level"))
display(published.limit(10))

### Human review of 50 facility–service pairs
Freeze the sample once. Sample across strong, moderate, weak and unmatched pairs, taking approximately equal numbers per available tier. The result measures agreement on this audit sample. It is not a population-weighted accuracy estimate. Unmatched pairs make missed evidence observable. The review table snapshots source text and predictions together and retains labels on reruns. Use a new review table name for a new rule version.

In [0]:
if not spark.catalog.tableExists(review_table):
    service_ids = spark.createDataFrame([(s,) for s in rules], "service string")
    pairs = (documents.select("unique_id").crossJoin(service_ids)
             .join(published.select("unique_id", "service", "evidence_level"),
                   ["unique_id", "service"], "left")
             .fillna({"evidence_level": "none"}))
    order = [F.xxhash64("unique_id", "service", F.lit(2026)), "unique_id", "service"]
    strata = Window.partitionBy("evidence_level").orderBy(*order)
    sample = (pairs.withColumn("pos", F.row_number().over(strata))
              .orderBy("pos", *order).limit(50).drop("pos")
              .withColumnRenamed("evidence_level", "prediction")
              .join(documents, "unique_id")
              .withColumn("manual_level", F.lit(None).cast("string")))
    sample.write.format("delta").mode("ignore").saveAsTable(review_table)
review = spark.table(review_table)
print("Frozen sample size:", review.count())
display(review.select("unique_id", "service", "source_records", "manual_level")
        .orderBy("unique_id", "service"))

Read all original source fields for each pair without looking at the prediction. Assign **strong** for explicit on-site service evidence, **moderate** for supporting equipment, **weak** for a relevant specialty alone, and **none** when there is no credible evidence or explicit unavailability. Use human judgment for wording, negation scope and conflicts. Record the independent judgment even when the rule may disagree.

Use a SQL cell to set each label in the frozen table, replacing the ID, service and level in this example.

```sql
UPDATE workspace.datathon.facility_service_review_v3
SET manual_level = 'moderate'
WHERE unique_id = '<actual ID>' AND service = 'dialysis'
```

After completing all 50, rerun the next cell. It calculates exact four-level agreement and shows the confusion matrix.

In [0]:
%sql
-- Human labels for the frozen v3 sample (50 pairs).
-- Correction after review: 8ed2e983 emergency_trauma changed from weak to strong
-- (its description states "offers 24x7 trauma care").
MERGE INTO workspace.datathon.facility_service_review_v3 AS t
USING (SELECT * FROM VALUES
  ('16f6abc2-90ff-4600-a92c-ac8c5057daf0', 'burn_care', 'none'),
  ('3b35b019-ec86-4a45-89a4-64c392a37775', 'c_section', 'none'),
  ('5803aea6-ecb9-4d97-818a-f9910816aca3', 'c_section', 'weak'),
  ('a385f373-09ae-494f-a9a3-af56f4181fd0', 'c_section', 'strong'),
  ('a91d2ef6-d9ea-4cad-96f8-87603aca54af', 'c_section', 'none'),
  ('f22a5809-fc26-4ee8-b4a7-5b12ed433465', 'c_section', 'weak'),
  ('26ab97e0-d7c6-482c-b043-87848a53da6a', 'cardiac_surgery', 'none'),
  ('2cb66ffa-8f3b-40be-9242-9431a9a96a5d', 'cardiac_surgery', 'strong'),
  ('6042140a-2c91-463f-86c8-8ef470d66c5b', 'cardiac_surgery', 'strong'),
  ('95e6fe70-5ce9-44bd-a221-d265cf089731', 'cardiac_surgery', 'none'),
  ('2e889e1d-ac65-4706-982c-a72ab71c7c36', 'ct_scan', 'weak'),
  ('553135af-070c-4f3c-81a0-96e810d4d0d8', 'ct_scan', 'moderate'),
  ('562f2b92-c90f-480d-bb93-e49a1e2285c0', 'ct_scan', 'weak'),
  ('7e48608f-183e-4af3-9c0e-60eb1f8c21e4', 'ct_scan', 'strong'),
  ('a1328542-caca-4116-929d-584e172b6a9c', 'ct_scan', 'moderate'),
  ('b8d2eb96-5ea2-4946-9d69-e12af15634ba', 'ct_scan', 'weak'),
  ('d59866af-483b-4aba-aa36-57ded325b48b', 'ct_scan', 'weak'),
  ('eceb9e5a-2311-4f06-acb5-914a383b14a1', 'ct_scan', 'moderate'),
  ('01b48e00-7414-4452-94b8-8506a25fb50b', 'dialysis', 'strong'),
  ('269e3a30-c0d8-447c-8d74-fe6a2977a315', 'dialysis', 'weak'),
  ('4c8ca503-236d-4c44-8a8e-9d31d5994f9e', 'dialysis', 'none'),
  ('51ee60c0-39ea-46ce-8f68-517cab365f19', 'dialysis', 'strong'),
  ('671afbc5-790d-4564-92f6-e68a94e4b2ec', 'dialysis', 'none'),
  ('8002c213-ef9f-42ec-bf62-532d95b94932', 'dialysis', 'weak'),
  ('dd0c2a24-cd0e-40d9-be45-adde9ab08272', 'dialysis', 'weak'),
  ('f9a2d9ba-4193-4e25-afb9-07dc5776f226', 'dialysis', 'weak'),
  ('2f340c9d-d246-446f-a70a-cfa49cdfbb37', 'emergency_trauma', 'strong'),
  ('4ddb163f-33d3-4925-93e2-df9c78ada503', 'emergency_trauma', 'none'),
  ('50ce83e9-a967-4274-90b7-c3cd4d56fd17', 'emergency_trauma', 'strong'),
  ('5ca2938c-7599-4c31-8a7b-ce039f38e0a8', 'emergency_trauma', 'strong'),
  ('8ed2e983-dadf-4caf-b9fa-821975ef8394', 'emergency_trauma', 'strong'),
  ('9c5537f1-5c81-4c7d-a7c9-12d606778780', 'emergency_trauma', 'none'),
  ('236b1b78-af76-4de6-abc1-8b6e5d03f9df', 'icu', 'strong'),
  ('26482460-2018-4bc7-bbab-2ec19f65308c', 'icu', 'none'),
  ('39c27df7-5e07-4626-8115-e7676e114de1', 'icu', 'moderate'),
  ('9960baf3-506c-435e-a525-d24a4dc82509', 'icu', 'strong'),
  ('d417844b-8d43-434a-82d4-a7ae6edf9109', 'icu', 'none'),
  ('d4225cd0-ff43-48a3-9d6c-eedef66ffa82', 'icu', 'none'),
  ('d93535a1-bda7-4c32-a3f9-a66f4efd7a47', 'icu', 'none'),
  ('068f6095-85f9-4d46-b3a7-b7209bfbd4c2', 'mri', 'strong'),
  ('12df23a8-087a-43c0-9ca7-e47d4937944e', 'mri', 'weak'),
  ('202dd36f-70dd-4802-9431-2925a8584560', 'mri', 'none'),
  ('3ad12409-c708-4acf-badd-4e25d6f2f1d5', 'mri', 'strong'),
  ('8b4d0b3f-896c-4010-8b7e-b5ebd781fb29', 'mri', 'strong'),
  ('8ed2d36d-231d-4d9c-b319-2b5e7de08bd6', 'mri', 'none'),
  ('c9768697-c29f-4e63-9deb-6d4cf6ddf0e9', 'mri', 'moderate'),
  ('decba67c-38f3-4130-9104-9038f062052e', 'mri', 'strong'),
  ('dfc174e8-db9a-4145-b009-299050e8daa6', 'mri', 'moderate'),
  ('8b0f4cdc-8e6e-43a1-b3d4-14dde039b97b', 'nicu', 'weak'),
  ('f52051e0-fbd4-4b16-bdf3-c1eb5ee67902', 'nicu', 'weak')
  AS v(unique_id, service, manual_level)) AS s
ON t.unique_id = s.unique_id AND t.service = s.service
WHEN MATCHED THEN UPDATE SET t.manual_level = s.manual_level;


In [0]:
review = spark.table(review_table)
valid_levels = ["strong", "moderate", "weak", "none"]
labelled = review.where(F.col("manual_level").isin(valid_levels))
n = review.count()
n_labelled = labelled.count()
if n == 50 and n_labelled == 50:
    correct = labelled.where(F.col("manual_level") == F.col("prediction")).count()
    print(f"Exact evidence-level accuracy on the audit sample: {correct}/50 = {correct / 50:.1%}")
    display(labelled.groupBy("manual_level").pivot("prediction", valid_levels)
            .count().fillna(0).orderBy("manual_level"))
else:
    print(f"{n_labelled} valid labels across {n} sampled pairs. Accuracy requires 50 reviewed pairs.")

In [0]:
%sql
SELECT service, manual_level AS human, prediction AS rule, unique_id
FROM workspace.datathon.facility_service_review_v3
WHERE manual_level <> prediction
ORDER BY service


In [0]:
current_docs = documents.select(
    "unique_id", F.col("source_records").alias("current_records")
)
current_scores = published.select(
    "unique_id", "service", F.col("evidence_level").alias("current_level")
)

check = (
    spark.table(review_table)
    .join(current_docs, "unique_id", "left")
    .join(current_scores, ["unique_id", "service"], "left")
    .withColumn("current_level", F.coalesce("current_level", F.lit("none")))
)

display(check.select(
    F.count("*").alias("sample_size"),
    F.count_if(~F.col("source_records").eqNullSafe(F.col("current_records")))
     .alias("source_mismatches"),
    F.count_if(~F.col("prediction").eqNullSafe(F.col("current_level")))
     .alias("prediction_mismatches")
))